In [1]:
import os
import csv
import random
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp
import matplotlib.pyplot as plt
from pathlib import Path
os.chdir(next(p for p in Path.cwd().parents if (p / 'src').exists()))

from src.networks.Encoder import Encoder
from src.networks.DecoderPINN import PINN
from src.data.encoder_dataloader import load_vae_inference_data
from src.hyperparameters import hypers_uqvae as h
from src.evaluation.encoder_eval import find_map, make_mcmc_and_hessian_fn, make_neg_log_post, make_gn_cov_fn, make_forward
from src.visualization.uqplots import propagate_and_plot_uncertainty_on_axes_sobol, plot_multiple_posterior_heatmaps

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.keras.backend.set_floatx("float32")
dtype = tf.float32

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

2026-10-04 17:40:25.784533: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-04 17:40:25.794944: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791128425.808162  262304 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791128425.813241  262304 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791128425.823501  262304 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
RESOLUTIONS = ["100x100", "200x200", "400x400", "800x800"]
EPS_COND = 1e10

# MAP (deterministic optimization independent of the encoder)
MAP_GRID = 3 # MAP_GRID x MAP_GRID starting points in [0.2, 0.8]^2
MAP_STEPS = 600
MAP_LR = 0.05
MAP_CHUNK = 50
# MCMC (only used for the acceptance-rate diagnostic, started at the MAP)
MCMC_STEPS = 300
MCMC_BURNIN = 100
MCMC_STEP_SIZE = 0.04

# quantiles: chi2(2 dof) 95% = -2 ln(0.05); normal 97.5%
CHI2_95_2DOF = -2.0 * np.log(0.05)
Z_95 = 1.959963984540054

SAMPLE_INDICES = [9, 27, 40, 41, 45, 80]
SAMPLE_INDICES_PROP = SAMPLE_INDICES[:5]

BASE_DATA_DIR = "data/multiresolution"
BASE_MODEL_DIR = "models/error_analysis"
BASE_RESULT_DIR = "results/error_analysis"
PLOT_DIR = f"{BASE_RESULT_DIR}/plots"

SENSOR_IDX_PATH = f"{BASE_DATA_DIR}/sensor_idx.npy"
CSV_PATH = f"{BASE_RESULT_DIR}/vae_vs_map_metrics.csv"

os.makedirs(PLOT_DIR, exist_ok=True)


def rel_fro(a, b):
    return np.linalg.norm(a - b, axis=(1, 2)) / (np.linalg.norm(b, axis=(1, 2)) + 1e-12)


def coverage(u_true, mu, cov):
    err = u_true - mu
    sd = np.sqrt(np.diagonal(cov, axis1=1, axis2=2))
    z = np.abs(err) / sd
    d2 = np.einsum('ni,nij,nj->n', err, np.linalg.inv(cov), err)
    return {"cx68": float(np.mean(z[:, 0] <= 1.0)), "cy68": float(np.mean(z[:, 1] <= 1.0)),
            "cx95": float(np.mean(z[:, 0] <= Z_95)), "cy95": float(np.mean(z[:, 1] <= Z_95)),
            "joint95": float(np.mean(d2 <= CHI2_95_2DOF))}


rows = []
header = (f"{'Grid':<10} | {'MAE VAE-MAP':<12} | {'MAE VAE-True':<13} | "
          f"{'MAE MAP-True':<13} | {'Cov Laplace Rel Err':<20} | "
          f"{'MCMC Acc %':<10} | {'Hess ok %':<10} | {'Newton step':<11}")
print(header)
print("-" * len(header))

for res in RESOLUTIONS:
    tf.keras.backend.clear_session()

    current_data_dir = f"{BASE_DATA_DIR}/{res}"
    current_model_dir = f"{BASE_MODEL_DIR}/{res}"
    current_plot_dir = f"{PLOT_DIR}/{res}"
    
    os.makedirs(current_plot_dir, exist_ok=True)

    lbfgs_weights_path = f"{current_model_dir}/decoder_lbfgs.weights.h5"
    encoder_weights_path = f"{current_model_dir}/encoder_adam.weights.h5"
    gamma_E_path = f"{current_model_dir}/gamma_E.npz"

    if not os.path.exists(lbfgs_weights_path):
        print(f"[{res}] Decoder weights not found at {lbfgs_weights_path}. Skipping.")
        continue
    if not os.path.exists(gamma_E_path):
        print(f"[{res}] {gamma_E_path} not found. Skipping.")
        continue

    # loading data
    y_test, u_test, X_obs, sensor_idx, U_SCALE = load_vae_inference_data(
        data_path=f"{current_data_dir}/test.npz",
        scale_path=f"{current_data_dir}/u_scale.npy",
        sensor_idx_path=SENSOR_IDX_PATH,
        dtype=dtype,
    )

    # models initialization
    decoder_pinn = PINN(h.N_NEURONS_DECODER, U_SCALE)
    decoder_pinn(tf.zeros((1, 4)))
    decoder_pinn.load_weights(lbfgs_weights_path)
    decoder_pinn.trainable = False

    encoder = Encoder(n_neurons=h.N_NEURONS_ENCODER, latent_dim=h.D_z)
    encoder(tf.zeros((1, h.N_SENSORS)))
    encoder.init_output_layer(bias=h.BIAS, weight_scale=h.WEIGHT_SCALE_ENCODER_OUT)
    encoder.load_weights(encoder_weights_path)

    # prior and noise
    mu_pr = tf.constant(h.MU_PR, dtype=dtype)
    sigma_pr = tf.constant(h.SIGMA_PR, dtype=dtype)
    Gamma_pr = tf.eye(h.D_z, dtype=dtype) * (sigma_pr ** 2)
    Gamma_pr_inv = tf.linalg.inv(Gamma_pr)
        
    gamma_data = np.load(gamma_E_path)
    mu_E = tf.constant(gamma_data['mu_E'], dtype=dtype)
    Gamma_E_inv = tf.constant(gamma_data['Gamma_E_inv'], dtype=dtype)

    # vae predictions
    qz_test = encoder(y_test)
    u_pred_test = qz_test.mean().numpy()
    Gamma_hat = qz_test.covariance() # (B, D_z, D_z)
    B = tf.shape(y_test)[0]
    Gamma_pr_inv_batch = tf.tile(Gamma_pr_inv[tf.newaxis, :, :], [B, 1, 1])
    
    # Gamma_post = Gamma_hat Gamma_pr^{-1} Gamma_hat
    Gamma_post = tf.matmul(Gamma_hat, tf.matmul(Gamma_pr_inv_batch, Gamma_hat))
    vae_cov_test = Gamma_post.numpy()

    # Gamma_post = A A^T with A = Gamma_hat L_pr^{-T}
    L_pr = tf.linalg.cholesky(Gamma_pr) # (2,2)
    A = tf.matmul(Gamma_hat, tf.linalg.inv(L_pr), transpose_b=True)
    qz_post = tfp.distributions.MultivariateNormalLinearOperator(
        loc=qz_test.mean(), scale=tf.linalg.LinearOperatorFullMatrix(A))

    # MAP estimation
    neg_log_post = make_neg_log_post(decoder_pinn, X_obs, mu_pr, Gamma_pr_inv, mu_E, Gamma_E_inv)
    u_map_test = find_map(neg_log_post=neg_log_post, y=tf.cast(y_test, dtype), dtype=dtype, n_grid=MAP_GRID, steps=MAP_STEPS, lr=MAP_LR, chunk=MAP_CHUNK)

    # MCMC (acceptance rate + exact-Hessian diagnostic only)
    mcmc_and_hessian = make_mcmc_and_hessian_fn(
        neg_log_post=neg_log_post, 
        dtype=dtype,
        mcmc_steps=MCMC_STEPS,
        mcmc_burnin=MCMC_BURNIN,
        mcmc_step_size=MCMC_STEP_SIZE,
        eps_cond=EPS_COND
    )

    # Laplace (J^T Gamma_E^-1 J + Gamma_pr^-1)^-1
    forward = make_forward(decoder_pinn, X_obs)
    gn_cov = make_gn_cov_fn(forward, Gamma_E_inv, Gamma_pr_inv)

    cov_gn_map, cov_gn_vae, hess_flags, acc_rates, newton_steps = [], [], [], [], []
    for i in range(len(y_test)):
        y_i = tf.cast(y_test[i], dtype)
        u_map_i = tf.constant(u_map_test[i], dtype=dtype)
        hess_ok, _, acc, ns = mcmc_and_hessian(u_map_i, y_i)
        cov_gn_map.append(gn_cov(u_map_i).numpy())
        cov_gn_vae.append(gn_cov(tf.constant(u_pred_test[i], dtype=dtype)).numpy())
        hess_flags.append(bool(hess_ok.numpy()))
        acc_rates.append(float(acc.numpy()))
        newton_steps.append(float(ns.numpy()))
    cov_gn_map, cov_gn_vae = np.array(cov_gn_map), np.array(cov_gn_vae)

    # metrics
    u_test_np = u_test.numpy() if hasattr(u_test, "numpy") else u_test
    mae_vae_map = float(np.mean(np.abs(u_pred_test - u_map_test)))
    mae_vae_true = float(np.mean(np.abs(u_pred_test - u_test_np)))
    mae_map_true = float(np.mean(np.abs(u_map_test - u_test_np)))

    err_map = rel_fro(vae_cov_test, cov_gn_map)
    err_vae = rel_fro(vae_cov_test, cov_gn_vae) 
    cov_error_laplace = float(np.mean(err_map))
    sd_v = np.sqrt(np.diagonal(vae_cov_test, axis1=1, axis2=2))
    sd_g = np.sqrt(np.diagonal(cov_gn_map, axis1=1, axis2=2))
    err_sd = float(np.mean(np.abs(sd_v - sd_g) / sd_g))

    pct_acc = 100.0 * float(np.mean(acc_rates))
    pct_hess = 100.0 * float(np.mean(hess_flags))
    median_newton = float(np.median(newton_steps))

    print(f"{res:<10} | {mae_vae_map:<12.4f} | {mae_vae_true:<13.4f} | "
          f"{mae_map_true:<13.4f} | {cov_error_laplace:<20.4e} | "
          f"{pct_acc:<10.1f} | {pct_hess:<10.1f} | {median_newton:<11.2e}")

    cov_post_stats = coverage(u_test_np, u_pred_test, vae_cov_test)
    cov_hat_stats = coverage(u_test_np, u_pred_test, Gamma_hat.numpy())
    print(res, "Γ_post:", cov_post_stats)
    print(res, "Γ_hat :", cov_hat_stats)

    rows.append({
        "Grid": res, 
        "mae_vae_map": mae_vae_map, 
        "mae_vae_true": mae_vae_true, 
        "mae_map_true": mae_map_true,
        "cov_laplace_rel_err": cov_error_laplace, 
        "cov_laplace_rel_err_median": float(np.median(err_map)),
        "cov_laplace_rel_err_at_vae": float(np.mean(err_vae)),
        "sd_rel_err": err_sd,
        "mcmc_acc_pct": pct_acc, 
        "hess_ok_pct": pct_hess, 
        "median_newton_step": median_newton, 
        **{f"post_{k}": v for k, v in cov_post_stats.items()},
        **{f"hat_{k}": v for k, v in cov_hat_stats.items()},
    })

    fig_prop, axes_prop = plt.subplots(5, 3, figsize=(18, 25))
    for i, idx in enumerate(SAMPLE_INDICES_PROP):
        propagate_and_plot_uncertainty_on_axes_sobol(
            decoder_pinn=decoder_pinn, 
            qz_val=qz_post, 
            u_pred_mean=u_pred_test, 
            u_true=u_test, 
            sample_idx=idx, 
            axes=axes_prop[i], 
            U_SCALE=float(U_SCALE), 
            K_samples=8192, 
            number_type=dtype
        )
        axes_prop[i][0].set_ylabel(f"Sample {idx}", fontsize=14, fontweight='bold', labelpad=20)
    
    plt.suptitle(f"Grid {res} - Spatial Uncertainty Propagation", fontsize=16, y=1.01)
    plt.tight_layout()
    fig_prop.savefig(f"{current_plot_dir}/uncertainty_propagation.pdf", dpi=300, bbox_inches='tight')
    plt.close(fig_prop)

    plot_multiple_posterior_heatmaps(
        qz_val=qz_post, 
        u_true=u_test, 
        sample_indices=SAMPLE_INDICES
    )
    fig_heat = plt.gcf()
    fig_heat.suptitle(f"Grid {res} - Posterior Density Heatmaps", fontsize=16)
    plt.tight_layout()
    fig_heat.savefig(f"{current_plot_dir}/posterior_heatmaps.pdf", dpi=300, bbox_inches='tight')
    plt.close(fig_heat)

if rows:
    with open(CSV_PATH, "w", newline="") as fcsv: 
        writer = csv.DictWriter(fcsv, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print(f"\nResults saved to: {CSV_PATH}")

Grid       | MAE VAE-MAP  | MAE VAE-True  | MAE MAP-True  | Cov Laplace Rel Err  | MCMC Acc % | Hess ok %  | Newton step
------------------------------------------------------------------------------------------------------------------------
U_SCALE (loaded): 0.06768616288900375


I0000 00:00:1791128429.916665  262304 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12305 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5080, pci bus id: 0000:01:00.0, compute capability: 12.0


Applied pre-computed consistent noise for inference.
Shape y_test: (101, 200)  |  Shape u_test: (101, 2)


I0000 00:00:1791128433.863062  262304 cuda_solvers.cc:175] Creating GpuSolver handles for stream 0x5a827a3dda10


100x100    | 0.0627       | 0.0438        | 0.0773        | 6.8730e-01           | 60.3       | 99.0       | 3.48e-04   
100x100 Γ_post: {'cx68': 0.33663366336633666, 'cy68': 0.43564356435643564, 'cx95': 0.5445544554455446, 'cy95': 0.594059405940594, 'joint95': 0.4158415841584158}
100x100 Γ_hat : {'cx68': 0.7623762376237624, 'cy68': 0.8118811881188119, 'cx95': 0.9108910891089109, 'cy95': 0.9405940594059405, 'joint95': 0.8910891089108911}
Uncertainty Propagation (Sobol) for sample 9 (True cx=0.595, cy=0.505)...
Uncertainty Propagation (Sobol) for sample 27 (True cx=0.522, cy=0.488)...
Uncertainty Propagation (Sobol) for sample 40 (True cx=0.368, cy=0.492)...
Uncertainty Propagation (Sobol) for sample 41 (True cx=0.414, cy=0.533)...
Uncertainty Propagation (Sobol) for sample 45 (True cx=0.507, cy=0.599)...
U_SCALE (loaded): 0.06774014234542847
Applied pre-computed consistent noise for inference.
Shape y_test: (101, 200)  |  Shape u_test: (101, 2)
200x200    | 0.0577       | 0.0516       